
# LSTM for Easy Text Prediction

## Dataset used
A **small custom text corpus** is used so that the lab:
- runs quickly
- is easy to understand
- does not require downloading datasets
- works well in most college lab environments



## 1. Import required libraries


In [47]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0



## 2. Create a small text dataset


In [57]:
corpus = [
    # unique endings per subject
    "deep learning is revolutionary",
    "machine learning is statistical",
    "artificial intelligence is futuristic",
    "lstm is sequential",
    "python is readable",
    "neural networks are layered",
    "transformers are attention based",
    "backpropagation updates network weights",
    "gradient descent minimizes the loss",
    "overfitting hurts model generalization",
    "dropout prevents model overfitting",
    "relu activates hidden neurons",
    "softmax outputs class probabilities",
    "embeddings represent words numerically",
    "tokenization splits text into tokens",
    "padding makes sequences equal length",
    "epochs define training iterations",
    "batches split training data",
    "loss measures prediction error",
    "accuracy measures correct predictions",
    "deep learning needs large datasets",
    "machine learning finds hidden patterns",
    "neural networks mimic human brain",
    "lstm remembers long sequences",
    "python simplifies complex algorithms",
    "convolutional networks process images",
    "recurrent networks process sequences",
    "attention mechanism improves predictions",
    "regularization reduces model complexity",
    "validation checks model generalization",
    "training teaches model parameters",
    "weights store learned information",
    "biases shift activation functions",
    "layers stack to build depth",
    "features represent input characteristics",
]

print("Number of sentences:", len(corpus))
print("\nSample corpus:")
for line in corpus:
    print("-", line)


Number of sentences: 35

Sample corpus:
- deep learning is revolutionary
- machine learning is statistical
- artificial intelligence is futuristic
- lstm is sequential
- python is readable
- neural networks are layered
- transformers are attention based
- backpropagation updates network weights
- gradient descent minimizes the loss
- overfitting hurts model generalization
- dropout prevents model overfitting
- relu activates hidden neurons
- softmax outputs class probabilities
- embeddings represent words numerically
- tokenization splits text into tokens
- padding makes sequences equal length
- epochs define training iterations
- batches split training data
- loss measures prediction error
- accuracy measures correct predictions
- deep learning needs large datasets
- machine learning finds hidden patterns
- neural networks mimic human brain
- lstm remembers long sequences
- python simplifies complex algorithms
- convolutional networks process images
- recurrent networks process sequen


## 3. Tokenization

A tokenizer converts words into integers.

Example:
- `deep` → 1
- `learning` → 2

The exact numbers depend on the fitted vocabulary.


In [58]:

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

word_index = tokenizer.word_index
total_words = len(word_index) + 1  # +1 because indexing starts from 1

print("Vocabulary size:", total_words)
print("\nWord index:")
print(word_index)


Vocabulary size: 112

Word index:
{'is': 1, 'model': 2, 'learning': 3, 'networks': 4, 'sequences': 5, 'training': 6, 'deep': 7, 'machine': 8, 'lstm': 9, 'python': 10, 'neural': 11, 'are': 12, 'attention': 13, 'weights': 14, 'loss': 15, 'overfitting': 16, 'generalization': 17, 'hidden': 18, 'represent': 19, 'measures': 20, 'predictions': 21, 'process': 22, 'revolutionary': 23, 'statistical': 24, 'artificial': 25, 'intelligence': 26, 'futuristic': 27, 'sequential': 28, 'readable': 29, 'layered': 30, 'transformers': 31, 'based': 32, 'backpropagation': 33, 'updates': 34, 'network': 35, 'gradient': 36, 'descent': 37, 'minimizes': 38, 'the': 39, 'hurts': 40, 'dropout': 41, 'prevents': 42, 'relu': 43, 'activates': 44, 'neurons': 45, 'softmax': 46, 'outputs': 47, 'class': 48, 'probabilities': 49, 'embeddings': 50, 'words': 51, 'numerically': 52, 'tokenization': 53, 'splits': 54, 'text': 55, 'into': 56, 'tokens': 57, 'padding': 58, 'makes': 59, 'equal': 60, 'length': 61, 'epochs': 62, 'define':


## 4. Create training sequences

For each sentence, we create multiple smaller sequences.

Example for:
`deep learning is powerful`

We create:
- `deep learning`
- `deep learning is`
- `deep learning is powerful`

Then the model learns:
- input = all words except last
- output = last word


In [59]:

input_sequences = []

for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print("Total training sequences:", len(input_sequences))
print("\nSome sequences before padding:")
for seq in input_sequences[:10]:
    print(seq)


Total training sequences: 110

Some sequences before padding:
[7, 3]
[7, 3, 1]
[7, 3, 1, 23]
[8, 3]
[8, 3, 1]
[8, 3, 1, 24]
[25, 26]
[25, 26, 1]
[25, 26, 1, 27]
[9, 1]



## 5. Pad the sequences

Not all sequences have the same length, so we pad them with zeros at the beginning.

This makes every input sequence have the same size.


In [60]:

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

print("Maximum sequence length:", max_seq_len)
print("\nPadded sequences:")
print(input_sequences[:10])


Maximum sequence length: 5

Padded sequences:
[[ 0  0  0  7  3]
 [ 0  0  7  3  1]
 [ 0  7  3  1 23]
 [ 0  0  0  8  3]
 [ 0  0  8  3  1]
 [ 0  8  3  1 24]
 [ 0  0  0 25 26]
 [ 0  0 25 26  1]
 [ 0 25 26  1 27]
 [ 0  0  0  9  1]]



## 6. Split into input (`X`) and output (`y`)

- `X` = all words except the last word
- `y` = the last word to be predicted

We convert `y` into one-hot encoding because this is a multiclass classification problem.


In [61]:

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


Shape of X: (110, 4)
Shape of y: (110, 112)



## 7. Build the LSTM model

### Model architecture
- **Embedding layer**: converts word indices into dense vectors
- **LSTM layer**: learns sequence patterns and context
- **Dense layer with softmax**: predicts the next word


In [62]:
model = Sequential([
    Embedding(input_dim=total_words, output_dim=64, input_length=max_seq_len - 1),
    LSTM(128, return_sequences=True, dropout=0.1),   # add this
    LSTM(128, dropout=0.1),                           # keep this
    Dense(64, activation='relu'),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), metrics=['accuracy'])
model.summary()


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


## 8. Train the model

Since the dataset is very small, we train for more epochs so the model can learn the patterns better.


In [69]:

history = model.fit(X, y, epochs=200, batch_size=8, verbose=1)

Epoch 1/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9091 - loss: 0.0774
Epoch 2/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9182 - loss: 0.0733
Epoch 3/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9636 - loss: 0.0688
Epoch 4/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9273 - loss: 0.0726
Epoch 5/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9455 - loss: 0.0714
Epoch 6/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9455 - loss: 0.0736
Epoch 7/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9545 - loss: 0.0729
Epoch 8/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9545 - loss: 0.0697
Epoch 9/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9273 - loss: 0.0725
Epoch 10/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9364 - loss: 0.0700
Epoch 11/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9364 - loss: 0.0716
Epoch 12/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


## 9. Function for next-word text generation

This function:
1. takes a seed text
2. converts it into tokens
3. pads it
4. predicts the next word
5. appends the predicted word to the sentence


In [70]:

def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text



## 10. Test the model

Try a few seed texts and observe the output.


In [71]:

print(generate_text("deep learning", 2))
print(generate_text("machine learning", 2))
print(generate_text("python is", 2))


deep learning needs large
machine learning finds hidden
python is readable information



## 11. Student activity section


In [72]:

# Try your own inputs here
print(generate_text("artificial intelligence", 2))
print(generate_text("data science", 2))
print(generate_text("lstm is", 3))


artificial intelligence is futuristic
data science mechanism makes
lstm is sequential numerically depth
